In [18]:
# Silevr Layer transformations


StatementMeta(, 635beed8-688c-429d-bde6-1a0571a9376b, 20, Finished, Available, Finished, False)

In [19]:
# Profile customer data
customers = spark.table("bronze_customers")

customers.printSchema()

StatementMeta(, 635beed8-688c-429d-bde6-1a0571a9376b, 21, Finished, Available, Finished, False)

root
 |-- customer_id: string (nullable = true)
 |-- first_name: string (nullable = true)
 |-- last_name: string (nullable = true)
 |-- email: string (nullable = true)
 |-- phone: string (nullable = true)
 |-- city: string (nullable = true)
 |-- country: string (nullable = true)
 |-- signup_date: date (nullable = true)
 |-- loyalty_tier: string (nullable = true)



In [20]:
display(customers.limit(10))

StatementMeta(, 635beed8-688c-429d-bde6-1a0571a9376b, 22, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 1fba8f47-2d43-4bf5-8e77-5f4c3917be5b)

In [21]:
# Count missing emails
from pyspark.sql.functions import col

customers.filter(
    col("email").isNull()
).count()

StatementMeta(, 635beed8-688c-429d-bde6-1a0571a9376b, 23, Finished, Available, Finished, False)

2000

In [22]:
# create silver customer
from pyspark.sql.functions import *

silver_customers = (
    customers
    .dropDuplicates(["customer_id"])
    .filter(col("email").isNotNull())
)

StatementMeta(, 635beed8-688c-429d-bde6-1a0571a9376b, 24, Finished, Available, Finished, False)

In [23]:
# validate 
silver_customers.count()

StatementMeta(, 635beed8-688c-429d-bde6-1a0571a9376b, 25, Finished, Available, Finished, False)

98000

In [24]:
# save
silver_customers.write \
    .mode("overwrite") \
    .format("delta") \
    .saveAsTable("silver_customers")

StatementMeta(, 635beed8-688c-429d-bde6-1a0571a9376b, 26, Finished, Available, Finished, False)

In [25]:
# read bronze 
orders = spark.table(
"bronze_orders"
)

StatementMeta(, 635beed8-688c-429d-bde6-1a0571a9376b, 27, Finished, Available, Finished, False)

In [26]:
from pyspark.sql.functions import current_date

orders.filter(
    col("order_date") > current_date()
).count()

StatementMeta(, 635beed8-688c-429d-bde6-1a0571a9376b, 28, Finished, Available, Finished, False)

500

In [27]:
# Create silver orders
silver_orders = (
    orders
    .filter(
        col("order_date") <= current_date()
    )
)

StatementMeta(, 635beed8-688c-429d-bde6-1a0571a9376b, 29, Finished, Available, Finished, False)

In [28]:
# save
silver_orders.write \
    .mode("overwrite") \
    .format("delta") \
    .saveAsTable("silver_orders")

StatementMeta(, 635beed8-688c-429d-bde6-1a0571a9376b, 30, Finished, Available, Finished, False)

In [29]:
# create solver products
products = spark.table(
    "bronze_products"
)

silver_products = (
    products
    .filter(
        col("selling_price") >
        col("cost_price")
    )
)

StatementMeta(, 635beed8-688c-429d-bde6-1a0571a9376b, 31, Finished, Available, Finished, False)

In [30]:
# save
silver_products.write \
    .mode("overwrite") \
    .format("delta") \
    .saveAsTable("silver_products")

StatementMeta(, 635beed8-688c-429d-bde6-1a0571a9376b, 32, Finished, Available, Finished, False)

In [31]:
# create silver stores
stores = spark.table(
    "bronze_stores"
)

silver_stores = stores.dropDuplicates(
    ["store_id"]
)

StatementMeta(, 635beed8-688c-429d-bde6-1a0571a9376b, 33, Finished, Available, Finished, False)

In [32]:
# save
silver_stores.write \
    .mode("overwrite") \
    .format("delta") \
    .saveAsTable("silver_stores")

StatementMeta(, 635beed8-688c-429d-bde6-1a0571a9376b, 34, Finished, Available, Finished, False)

In [33]:
# create silver order_items
order_items = spark.table(
    "bronze_order_items"
)

silver_order_items = (
    order_items
    .filter(col("quantity") > 0)
)

StatementMeta(, 635beed8-688c-429d-bde6-1a0571a9376b, 35, Finished, Available, Finished, False)

In [34]:
# save 
silver_order_items.write \
    .mode("overwrite") \
    .format("delta") \
    .saveAsTable("silver_order_items")

StatementMeta(, 635beed8-688c-429d-bde6-1a0571a9376b, 36, Finished, Available, Finished, False)

In [35]:
# create silver returns 
returns = spark.table(
    "bronze_returns"
)

silver_returns = (
    returns
    .filter(
        col("refund_amount") > 0
    )
)

StatementMeta(, 635beed8-688c-429d-bde6-1a0571a9376b, 37, Finished, Available, Finished, False)

In [36]:
# save 
silver_returns.write \
    .mode("overwrite") \
    .format("delta") \
    .saveAsTable("silver_returns")

StatementMeta(, 635beed8-688c-429d-bde6-1a0571a9376b, 38, Finished, Available, Finished, False)

In [37]:
# silver tables validation
tables = [
    "silver_customers",
    "silver_products",
    "silver_stores",
    "silver_orders",
    "silver_order_items",
    "silver_returns"
]

for table in tables:
    cnt = spark.sql(
        f"SELECT COUNT(*) cnt FROM {table}"
    ).collect()[0]["cnt"]

    print(f"{table}: {cnt:,}")

StatementMeta(, 635beed8-688c-429d-bde6-1a0571a9376b, 39, Finished, Available, Finished, False)

silver_customers: 98,000
silver_products: 5,000
silver_stores: 100
silver_orders: 499,500
silver_order_items: 600,031
silver_returns: 25,000
